# Tests 

In [ ]:
import torch

def _generate_sequence_mask(length, mask_current=False):
        """
        Génère un masque de séquence pour le modèle Transformer.

        Paramètres :
        - length (int) : Taille de la séquence.
        - see_current (bool) : Indique si le modèle peut voir l'élement actuel.

        Retourne :
        - torch.Tensor : Masque de séquence.
        """
        if mask_current:
            length += 1
        mask = (torch.triu(torch.ones(length, length)) == 1).transpose(0, 1)
        mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
        mask = mask.to(torch.device('mps'))
        return mask if not mask_current else mask[:-1, 1:]

_generate_sequence_mask(5, mask_current=True)

## Test Benchmark

In [ ]:
from src.benchmark import *
from src.tasks import *
from RAU import RAU

# Création du benchmark
seeds = [1, 2, 3, 4]
benchmark = Benchmark(model_class=RAU, model_name="RAU", seeds=seeds)
n_trials = 2

# Ajout des tâches
benchmark.add_task(Task(
    name="discrete_postcasting",
    generator=generate_discrete_postcasting,
    is_classification=True,
    model_args={"input_dim": 10, "output_dim": 10, "units": [10, 500], "degree": [1, 3], 'spectral_radius': [0., 1.], 'leak_rate': [0., 1.]},
    generator_params={"sequence_length": 1000, "delay": 10, "n_symbols": 10},
    n_trials=n_trials
))

benchmark.add_task(Task(
    name="continue_postcasting",
    generator=generate_continue_postcasting,
    is_classification=False,
    model_args={"input_dim": 1, "output_dim": 1, "units": [10, 500], "degree": 1, 'spectral_radius': [0., 1.], 'leak_rate': [0., 1.]},
    generator_params={"sequence_length": 1000, "delay": 10},
    n_trials=n_trials
))

benchmark.add_task(Task(
    name="copy_task",
    generator=generate_copy_task,
    is_classification=True,
    model_args={"input_dim": 11, "output_dim": 10, "units": [10, 500], "degree": [1, 11], 'spectral_radius': [0., 1.], 'leak_rate': [0., 1.]},
    generator_params={"sequence_length": 100, "delay": 10, "n_symbols": 10},
    n_trials=n_trials
))

benchmark.add_task(Task(
    name="selective_copy_task",
    generator=generate_selective_copy_task,
    is_classification=True,
    model_args={"input_dim": 12, "output_dim": 10, "units": [10, 500], "degree": [1, 10], 'spectral_radius': [0., 1.], 'leak_rate': [0., 1.]},
    generator_params={"sequence_length": 100, "delay": 10, "n_symbols": 10},
    n_trials=n_trials
))

benchmark.add_task(Task(
    name="adding_problem",
    generator=generate_adding_problem,
    is_classification=True,
    model_args={"input_dim": 11, "output_dim": 17, "units": [10, 500], "degree": [1, 11], 'spectral_radius': [0., 1.], 'leak_rate': [0., 1.]},
    generator_params={"sequence_length": 50, "max_number": 9},
    n_trials=n_trials
))

benchmark.add_task(Task(
    name="sorting_problem",
    generator=generate_sorting_problem,
    is_classification=True,
    model_args={"input_dim": 61, "output_dim": 10, "units": [10, 500], "degree": [1, 10], 'spectral_radius': [0., 1.], 'leak_rate': [0., 1.]},
    generator_params={"sequence_length": 50, "n_symbols": 10},
    n_trials=n_trials
))

benchmark.add_task(Task(
    name="mnist_classification",
    generator=generate_mnist_classification,
    is_classification=True,
    model_args={"input_dim": 29, "output_dim": 10, "units": [10, 500], "degree": [1, 10], 'spectral_radius': [0., 1.], 'leak_rate': [0., 1.]},
    generator_params={},
    n_trials=n_trials
))

benchmark.add_task(Task(
    name="discrete_pattern_completion",
    generator=generate_discrete_pattern_completion,
    is_classification=True,
    model_args={"input_dim": 9, "output_dim": 8, "units": [10, 500], "degree": [1, 9], 'spectral_radius': [0., 1.], 'leak_rate': [0., 1.]},
    generator_params={"sequence_length": 100, "n_symbols": 8, "base_length": 10},
    n_trials=n_trials
))

benchmark.add_task(Task(
    name="continue_pattern_completion",
    generator=generate_continue_pattern_completion,
    is_classification=False,
    model_args={"input_dim": 1, "output_dim": 1, "units": [10, 500], "degree": 1, 'spectral_radius': [0., 1.], 'leak_rate': [0., 1.]},
    generator_params={"sequence_length": 100, "base_length": 10},
    n_trials=n_trials
))

benchmark.add_task(Task(
    name="bracket_matching",
    generator=generate_bracket_matching,
    is_classification=True,
    model_args={"input_dim": 3, "output_dim": 1, "units": [10, 500], "degree": [1, 3], 'spectral_radius': [0., 1.], 'leak_rate': [0., 1.]},
    generator_params={"sequence_length": 50, "max_depth": 5},
    n_trials=n_trials
))

benchmark.add_task(Task(
    name="sin_forecasting",
    generator=generate_sin_forecasting,
    is_classification=False,
    model_args={"input_dim": 1, "output_dim": 1, "units": [10, 500], "degree": 1, 'spectral_radius': [0., 1.], 'leak_rate': [0., 1.]},
    generator_params={"sequence_length": 10000, "forecast_length": 10},
    n_trials=n_trials
))

benchmark.add_task(Task(
    name="chaotic_forecasting",
    generator=generate_chaotic_forecasting,
    is_classification=False,
    model_args={"input_dim": 3, "output_dim": 3, "units": [10, 500], "degree": [1, 3], 'spectral_radius': [0., 1.], 'leak_rate': [0., 1.]},
    generator_params={"sequence_length": 10000, "forecast_length": 10},
    n_trials=n_trials
))


# Evaluation du modèle
benchmark.run()

# Génération du rapport
benchmark.generate_report()

## Test LSTM

In [ ]:
import numpy as np
from src.models.LSTM import LSTM
from src.tasks import generate_discrete_postcasting

# Generate dataset
X_train, Y_train, X_test, Y_test = generate_discrete_postcasting(100, 10, 10)

# Create & Train model
model = LSTM(hidden_size=20, num_layers=2, learning_rate=1e-3, device="mps")
model.train(X_train, Y_train, epochs=10, batch_size=10, classification=True)

# Evaluate model
Y_preds = model.run(X_test)
Y_preds = (Y_preds > 0.5).astype(int)

# Compute accuracy
accuracy = np.mean(Y_preds == Y_test)

print("Number of parameter:", model.count_params())
print("Accuracy:", accuracy)


## Test Transformers Encoder-Decoder

In [ ]:
from src.models.Transformers import Transformers
from src.tasks import generate_discrete_postcasting, generate_copy_task
import numpy as np

# Generate dataset
# X_train, Y_train, X_test, Y_test = generate_discrete_postcasting(sequence_length=1000, delay=10, n_symbols=10)
X_train, Y_train, X_test, Y_test = generate_copy_task(n_samples=10, sequence_length=20, delay=2, n_symbols=3)

# Create & Train model
model_args = {
    "d_model": 16,
    "nhead": 2,
    "num_encoder_layers": 8,
    "num_decoder_layers": 16,
    "dim_feedforward": 64,
    "dropout": 0.1,
    "device": "cpu",
}
model = Transformers(**model_args)
model.train(X_train, Y_train, epochs=10, batch_size=10, classification=True)
Y_preds = model.run(X_test).cpu().numpy()
Y_preds = (Y_preds > 0.5).astype(int)

# Compute accuracy
accuracy = np.mean(Y_preds == Y_test)

print("Number of parameter:", model.count_params())
print("Accuracy:", accuracy)

## Test Transformer-Decoder-Only

In [ ]:
from src.models.TransformerDecoderOnly import TransformerDecoderOnly
from src.tasks import generate_discrete_postcasting, generate_copy_task
import numpy as np

# Generate dataset
X_train, Y_train, X_test, Y_test = generate_copy_task(n_samples=10, sequence_length=20, delay=2, n_symbols=3)

# Create & Train model
model_args = {
    "d_model": 16,
    "nhead": 2,
    "num_encoder_layers": 8,
    "num_decoder_layers": 16,
    "dim_feedforward": 64,
    "dropout": 0.1,
    "device": "cpu",
}
model = TransformerDecoderOnly()
model.train(X_train, Y_train, epochs=10, batch_size=10, classification=True)
Y_preds = model.run(X_test).cpu().numpy()
Y_preds = (Y_preds > 0.5).astype(int)

# Compute accuracy
accuracy = np.mean(Y_preds == Y_test)

print("Number of parameter:", model.count_params())
print("Accuracy:", accuracy)

## Test ESN

In [ ]:
import numpy as np
from src.models.ESN import ESN
from src.tasks import generate_discrete_postcasting, generate_chaotic_forecasting

# Generate dataset
X_train, Y_train, X_test, Y_test = generate_discrete_postcasting(1000, 10, 10)
#X_train, Y_train, X_test, Y_test = generate_chaotic_forecasting(10000, 10)

# Create & Train model
model = ESN(n_units=500, spectral_radius=0.9, leak_rate=1)
model.train(X_train, Y_train)

# Evaluate model
Y_preds = model.run(X_test)
Y_preds = (Y_preds > 0.5).astype(int)

# Compute accuracy
accuracy = np.mean(Y_preds == Y_test)
mse = np.mean((Y_preds - Y_test) ** 2)

print("Number of parameter:", model.count_params())
print("Accuracy:", accuracy)
print("MSE:", mse)

In [ ]:
from reservoirpy.nodes import Reservoir, Ridge
from src.tasks import generate_copy_task
import numpy as np
import reservoirpy as rpy

rpy.verbosity(0)


# Generate dataset
X_train, Y_train, X_test, Y_test = generate_copy_task(n_samples=10, sequence_length=20, delay=2, n_symbols=3)

# Generate model
res = Reservoir(100, sr=0.9, lr=0.7)
ridges = [Ridge(ridge=0.1**i) for i in range(1, 11)]

# Run reservoir
states = []
for i in range(X_train.shape[0]):
    states.append(res.run(X_train[i]))
states = np.array(states)

# Train ridges
errors = []
for ridge in ridges:
    # Train ridge
    ridge.fit(states, Y_train)

    # Make prediction on train dataset
    y_preds = []
    for i in range(X_train.shape[0]):
        y_preds += [ridge.run(states[i])]
    y_preds = np.array(y_preds)

    # Compute errors on train dataset
    errors += [np.mean((y_preds - Y_train[i]) ** 2)]

# Create best model
ridge = ridges[np.argmin(errors)]
model = res >> ridge
model

## Explore Results Dataframe

In [ ]:
import pandas as pd

df = pd.read_csv("results/ESN-test/results.csv")

# Suppression des colonnes inutiles
dfnp = df.drop(['Task Args', 'Model Args', 'Training Args'], axis=1)

# Création des groupes par 'Task' et 'Model'
groups = dfnp.groupby(['Task', 'Model'])

# Initialisation d'une liste pour les indices des meilleurs modèles
best_indices = []

# Parcours de chaque groupe
for (task, model), group in groups:
    # Trier par performance décroissante (remplacez 'Performance' par votre métrique)
    sorted_group = group.sort_values(by='MSE', ascending=True)
    
    # Obtenir les 10% meilleurs modèles
    top_5_percent = sorted_group.head(max(1, int(len(sorted_group) * 0.05)))  # Au moins 1 modèle
    
    # Trouver l'indice du modèle avec le BIC minimum dans les 10% supérieurs
    best_idx = top_5_percent['BIC'].idxmin()
    
    # Ajouter cet indice à la liste
    best_indices.append(best_idx)

# Récupérer les meilleurs modèles en fonction des indices
best_models = dfnp.loc[best_indices]

top_10_percent


## Generate README Summary Table

In [73]:
import os
import pandas as pd 
import numpy as np

# Récupération des résultats
folder = "baseline"
models = os.listdir(folder)
models.sort()

# Création d'une liste pour stocker les DataFrames
dfs = []
for model in models:
    path = os.path.join(folder, model)
    csvs = [elem for elem in os.listdir(path) if elem.split('.')[-1] == 'csv']
    for csv in csvs:
        df = pd.read_csv(os.path.join(path, csv))
        df['Model'] = model
        dfs.append(df)

# Concaténation des DataFrames
df = pd.concat(dfs).reset_index(drop=True)

# Ajout de la colonne 'Error'
df['Error'] = df.apply(lambda x: x['MSE'] if not np.isnan(x['MSE']) else 1-x['Accuracy'], axis=1)

# Suppression des colonnes inutiles
usefull_columns = ['Model', 'Task', 'Error', 'Number Params', 'Model Args', 'Task Args', 'Training Args']
df = df[usefull_columns]

# Define the ranges of parameters to consider
max_params = [1e3, 1e4, 1e5, 1e6, np.inf]
max_names = ['1k', '10k', '100k', '1M', 'inf']

# Init markdown with header
tab_markdown = f"| Task | {' |||||| '.join(models)} ||||||\n"
tab_markdown += "|-|" + "-|-|-|-|-|-|" * len(models) + "\n"
param_ranges = [f"<{name}" for name in max_names]
header_row = "|| " + " || ".join([" | ".join(param_ranges) for _ in models]) + " |\n"
tab_markdown += header_row

# For each tasks
tasks = df['Task'].unique()
for task in tasks:
    # Select results for the task
    df_task = df[df['Task'] == task]

    # Init row markdown
    row = f"| {task} |"

    # Extract best errors
    errors = {}
    for model in models:
        # Select results for the model
        df_model = df_task[df_task['Model'] == model]
        errors[model] = []

        # Extract best result for each max number of parameters
        min_param = 0
        for i, max_param in enumerate(max_params):
            df_max_param = df_model[(df_model['Number Params'] <= max_param) & (df_model['Number Params'] > min_param)]
            min_param = max_param
            errors[model] += [df_max_param['Error'].min() if len(df_max_param) > 0 else None]

    # Display errors
    for model in models:
        min_error = min([error for error in errors[model] if error is not None])
        # Add errors to the row
        for error in errors[model]:
            if error is not None:
                row += f" **{error:.3f}** |" if error == min_error else f" {error:.3f} |"
            else:
                row += " - |"
        row += "|"

    row += "\n"  # End the row
    tab_markdown += row

# Output the markdown table
print(tab_markdown)

| Task | ESN |||||| LSTM |||||| TransformerDecoderOnly ||||||
|-|-|-|-|-|-|-|-|-|-|-|-|-|-|-|-|-|-|-|
|| <1k | <10k | <100k | <1M | <inf || <1k | <10k | <100k | <1M | <inf || <1k | <10k | <100k | <1M | <inf |
| copy_task | 1.000 | **0.984** | 0.997 | - | - || 0.993 | 0.892 | **0.858** | 0.864 | 0.890 || - | 0.893 | **0.891** | 0.899 | - ||
| discrete_postcasting | 1.000 | 0.958 | **0.916** | - | - || - | **0.963** | 0.968 | 0.984 | 0.995 || - | 0.968 | **0.942** | 0.968 | - ||
| continue_pattern_completion | 0.015 | 0.014 | **0.012** | - | - || 0.014 | 0.008 | 0.003 | 0.002 | **0.002** || - | 0.015 | 0.015 | **0.015** | - ||
| selective_copy_task | 1.000 | **0.980** | 0.995 | - | - || 0.946 | 0.895 | 0.892 | **0.890** | 0.891 || - | 0.891 | **0.890** | 0.892 | - ||
| sorting_problem | 1.000 | 0.987 | **0.984** | - | - || - | 0.903 | 0.896 | 0.893 | **0.892** || - | **0.892** | **0.892** | **0.892** | - ||
| chaotic_forecasting | **20.106** | 38.106 | 658.972 | - | - || 247.040 | 218.27

In [70]:
errors

{'ESN': [0.5549999999999999, 0.21999999999999997, 0.255, None, None],
 'LSTM': [None, 1.0, 1.0, 1.0, 1.0],
 'TransformerDecoderOnly': [None, 0.89, 0.84, 0.84, None]}

[0.5549999999999999,
 0.21999999999999997,
 0.255,
 1.0,
 1.0,
 1.0,
 1.0,
 0.89,
 0.84,
 0.84]

## Test all tasks generators

#### Test simple memory

In [16]:
#np.set_printoptions(precision=np.inf)

In [1]:
from src.tasks import * 

X_train, Y_train, T_train, X_test, Y_test, T_test = generate_discrete_postcasting(1000, 10, 3)
print(X_train.shape, Y_train.shape, T_train.shape, X_test.shape, Y_test.shape, T_test.shape)

(1, 800, 3) (1, 800, 3) (1, 790) (1, 200, 3) (1, 200, 3) (1, 190)


/Users/naowak/Thesis/code/DistilledTransformerReservoir/dtrvenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from src.tasks import * 

X_train, Y_train, T_train, X_test, Y_test, T_test = generate_continue_postcasting(1000, 10)
print(X_train.shape, Y_train.shape, T_train.shape, X_test.shape, Y_test.shape, T_test.shape)

(1, 800, 1) (1, 800, 1) (1, 790) (1, 200, 1) (1, 200, 1) (1, 190)


In [1]:
from src.tasks import * 

X_train, Y_train, T_train, X_test, Y_test, T_test = generate_copy_task(100, 10, 2, 3)
print(X_train.shape, Y_train.shape, T_train.shape, X_test.shape, Y_test.shape, T_test.shape)

(80, 23, 4) (80, 23, 3) (80, 10) (20, 23, 4) (20, 23, 3) (20, 10)


/Users/naowak/Thesis/code/DistilledTransformerReservoir/dtrvenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from src.tasks import * 

X_train, Y_train, T_train, X_test, Y_test, T_test = generate_selective_copy_task(100, 10, 2, 3, 3)
print(X_train.shape, Y_train.shape, T_train.shape, X_test.shape, Y_test.shape, T_test.shape)

(80, 16, 5) (80, 16, 3) (80, 3) (20, 16, 5) (20, 16, 3) (20, 3)


In [3]:
from src.tasks import * 

X_train, Y_train, T_train, X_test, Y_test, T_test = generate_adding_problem(10, 10, 3)
print(X_train.shape, Y_train.shape, T_train.shape, X_test.shape, Y_test.shape, T_test.shape)

(8, 12, 5) (8, 12, 5) (8, 1) (2, 12, 5) (2, 12, 5) (2, 1)


In [5]:
from src.tasks import * 

X_train, Y_train, T_train, X_test, Y_test, T_test = generate_sorting_problem(10, 4, 3)
print(X_train.shape, Y_train.shape, T_train.shape, X_test.shape, Y_test.shape, T_test.shape)


(8, 9, 8) (8, 9, 3) (8, 4) (2, 9, 8) (2, 9, 3) (2, 4)


In [2]:
from src.tasks import * 

X_train, Y_train, T_train, X_test, Y_test, T_test = generate_mnist_classification(10, 0.8)
print(X_train.shape, Y_train.shape, T_train.shape, X_test.shape, Y_test.shape, T_test.shape)


(8, 30, 29) (8, 30, 10) (8, 1) (2, 30, 29) (2, 30, 10) (2, 1)


In [5]:
from src.tasks import * 

X_train, Y_train, T_train, X_test, Y_test, T_test = generate_discrete_pattern_completion(20, 20, 4)
print(X_train.shape, Y_train.shape, T_train.shape, X_test.shape, Y_test.shape, T_test.shape)


(16, 20, 5) (16, 20, 4) (16, 4) (4, 20, 5) (4, 20, 4) (4, 4)


In [7]:
from src.tasks import * 

X_train, Y_train, T_train, X_test, Y_test, T_test = generate_continue_pattern_completion(20, 20, 3)
print(X_train.shape, Y_train.shape, T_train.shape, X_test.shape, Y_test.shape, T_test.shape)

(16, 20, 1) (16, 20, 1) (16, 4) (4, 20, 1) (4, 20, 1) (4, 4)


In [8]:
from src.tasks import * 

X_train, Y_train, T_train, X_test, Y_test, T_test = generate_bracket_matching(20, 10)
print(X_train.shape, Y_train.shape, T_train.shape, X_test.shape, Y_test.shape, T_test.shape)

(16, 12, 3) (16, 12, 1) (16, 1) (4, 12, 3) (4, 12, 1) (4, 1)
[11] [11]


In [11]:
from src.tasks import * 

X_train, Y_train, T_train, X_test, Y_test, T_test = generate_sin_forecasting(20, 2)
print(X_train.shape, Y_train.shape, T_train.shape, X_test.shape, Y_test.shape, T_test.shape)

(1, 16, 1) (1, 16, 1) (1, 14) (1, 4, 1) (1, 4, 1) (1, 2)


In [12]:
from src.tasks import * 

X_train, Y_train, T_train, X_test, Y_test, T_test = generate_chaotic_forecasting(20, 2)
print(X_train.shape, Y_train.shape, T_train.shape, X_test.shape, Y_test.shape, T_test.shape)

(1, 16, 3) (1, 16, 3) (1, 14) (1, 4, 3) (1, 4, 3) (1, 2)


In [ ]:
import src.tasks as tasks
import seaborn as sns

X_train, Y_train, T_train, X_test, Y_test, T_test = tasks.generate_sin_forecasting(100, 1, 0.8)
print(X_train.shape, Y_train.shape, T_train.shape, X_test.shape, Y_test.shape, T_test.shape)

sns.lineplot(x=range(80), y=X_train[0, :, 0])
sns.lineplot(x=range(80), y=Y_train[0, :, 0])

In [ ]:
import src.tasks as tasks
from plotly import express as px

X_train, Y_train, T_train, X_test, Y_test, T_test = tasks.generate_chaotic_forecasting(10000, 1, 0.8)
print(X_train.shape, Y_train.shape, T_train.shape, X_test.shape, Y_test.shape, T_test.shape)

px.line_3d(x=X_train[0, :, 0], y=X_train[0, :, 1], z=X_train[0, :, 2])